# RKNN 변환 환경 (YOLOv8n → RK3588 INT8)

- 목적: Colab(x86_64 Linux)에서 **RKNN-Toolkit2**를 설치하고 YOLOv8n ONNX를 RK3588용 `.rknn`으로 변환한다. (PLAN BOARD-06)
- 런타임: CPU로 충분 (GPU 불필요)
- **새 런타임에서 실행**: rkllm-toolkit과 의존성(torch·protobuf 버전)이 충돌하므로 RKLLM 노트북과 같은 세션에서 돌리지 않는다.
- 절차·근거: `recipes/yolo-to-rknn.md`
- 끝나면: 출력의 버전·시간을 이슈 #13에 댓글로 남긴다.

In [ ]:
import platform
import sys

print(sys.version)
print(platform.machine())  # x86_64 이어야 한다

In [ ]:
# JARVIS 저장소 (변환 스크립트)
!git clone -q https://github.com/samsung-team-jarvis/jarvis.git /content/jarvis || (cd /content/jarvis && git pull -q)

In [ ]:
# RKNN-Toolkit2: 공식 저장소의 버전 고정 requirements → 휠 순서로 설치
RKNN_VERSION = "2.3.2"
import sys

tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
base = f"https://raw.githubusercontent.com/airockchip/rknn-toolkit2/v{RKNN_VERSION}/rknn-toolkit2/packages/x86_64"
req = f"{base}/requirements_{tag}-{RKNN_VERSION}.txt"
whl = f"{base}/rknn_toolkit2-{RKNN_VERSION}-{tag}-{tag}-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
!pip install -q -r {req}
!pip install -q {whl}

설치 후 import 오류가 나면 **런타임 → 세션 다시 시작** 후 아래 셀부터 실행한다.

In [ ]:
from rknn.api import RKNN

print("rknn-toolkit2 import OK")

In [ ]:
# 샘플 모델: rknn_model_zoo가 배포하는 YOLOv8n ONNX + COCO 캘리브레이션 20장
!git clone -q --depth 1 https://github.com/airockchip/rknn_model_zoo.git /content/rknn_model_zoo || true
!cd /content/rknn_model_zoo/examples/yolov8/model && bash download_model.sh && ls -lh *.onnx

In [ ]:
!mkdir -p /content/out
!python /content/jarvis/training/convert/yolo_onnx_to_rknn.py \
    --onnx /content/rknn_model_zoo/examples/yolov8/model/yolov8n.onnx \
    --dataset /content/rknn_model_zoo/datasets/COCO/coco_subset_20.txt \
    --out /content/out/yolov8n_sample_i8.rknn --dtype i8

## 우리 모델로 바꾸려면

1. `recipes/yolo-to-rknn.md` 방법 A로 학습한 모델을 ONNX로 export
2. 캘리브레이션 목록(`dataset.txt`): **Train 세트의 주방 사진** 경로를 한 줄에 하나씩
3. 위 셀의 `--onnx`, `--dataset`, `--out`만 바꿔 실행

In [ ]:
# (선택) 결과를 팀 Drive에 저장. 가중치는 git에 올리지 않는다.
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/jarvis/models"
!cp -v /content/out/* "/content/drive/MyDrive/jarvis/models/"